In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-01 13:32:12 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\data
2026-04-01 13:32:12 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [3]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-01 13:32:14 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1315 ερωτήσεις-απαντήσεις!


In [4]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices", [])
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type

In [5]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "test_dataset.xlsx"

df = pd.DataFrame(main_dataset)
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

2026-04-01 13:32:19 - INFO - Tο αρχείο δημιουργήθηκε επιτυχώς στο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\results\test_dataset.xlsx!


,id,question,input,choices,images,mark,answer,subject,year,school_type,format
0,Α1.α.1,Ποια είναι η κύρια αιτία η οποία εμποδίζει του...,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. Οι αλυσίδες στους αυχένες τους., β. Το σκο...",[],[Μονάδες 2],α,arxaia,2025,GEL,multiple_choice
1,Α1.α.2,Πού βρίσκεται το πυρ σε σχέση με τους δεσμώτες;,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. Μπροστά τους, χαμηλά., β. Επάνω και πίσω τ...",[],[Μονάδες 2],β,arxaia,2025,GEL,multiple_choice
2,Α1.α.3,"Τα «σκεύη», οι «ἀνδριάντες» και τα «ἄλλα ζῷα» ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,"[α. είναι πραγματικά ζώα της σπηλιάς., β. δημι...",[],[Μονάδες 2],β,arxaia,2025,GEL,multiple_choice
3,Α1.β,"«παρ’ ἣν», «ὑπὲρ ὧν»: Σε ποια λέξη του αρχαίου...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],[Μονάδες 4],παρ ́ἥν: αναφέρεται στην ὁδόν\nὑπέρ ὧν: αναφέρ...,arxaia,2025,GEL,open_ended
4,B1,"Ποιος είναι ο βασικός εκφραστικός τρόπος, με τ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],[Μονάδες 10],Ο κυριότερος εκφραστικός τρόπος με τον οποίο ο...,arxaia,2025,GEL,open_ended
